# Hands-On 5: Build a Chatbot with Memory

We add persistence with checkpointers and thread IDs so the chatbot remembers conversations across sessions.

#### Imports

We import `MessagesState` for message-based state, `InMemorySaver` as the checkpointer, and message types from LangChain.

In [4]:
import os

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

In [5]:
load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter API key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))

OpenRouter API key loaded: True


#### Create Conversation Graph

A simple echo assistant that stores messages in state. The assistant node reads the last message and replies with a message counter.

In [6]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter LLM configured successfully")

OpenRouter LLM configured successfully


In [7]:
response = llm.invoke("Hello, introduce yourself in one sentence.")

print(response.content)

Hello! I'm an AI language model designed to assist with a wide range of questions and tasks by providing information and generating text based on your input.


In [8]:
def assistant_node(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

#### Configure Checkpointer

A checkpointer saves the state after every step, enabling persistence across invocations.

In [9]:
checkpointer = InMemorySaver()

print("Checkpointer configured successfully")

Checkpointer configured successfully


#### Build Graph with Checkpointer

We create a simple graph with one assistant node and compile it with the checkpointer.

In [10]:
graph = StateGraph(MessagesState)

graph.add_node("assistant", assistant_node)

graph.add_edge(START, "assistant")
graph.add_edge("assistant", END)

chatbot = graph.compile(
    checkpointer=checkpointer
)

print("Conversation graph compiled successfully")

Conversation graph compiled successfully


#### Create Thread ID

A thread ID identifies a conversation. Each thread has its own isolated memory.

In [11]:
config = {
    "configurable": {
        "thread_id": "session-1"
    }
}

print("Thread ID:", config["configurable"]["thread_id"])

Thread ID: session-1


#### Store Conversation State

Each invoke with `config` saves the state under that thread ID, building up conversation history.

In [12]:
result1 = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="My name is Thoufeeq.")
        ]
    },
    config=config
)

print("AI:", result1["messages"][-1].content)

AI: Nice to meet you, Thoufeeq! How can I assist you today?


#### Ask Follow-Up Questions

The bot now has access to the full conversation history stored in the thread.

In [13]:
result2 = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?")
        ]
    },
    config=config
)

print("AI:", result2["messages"][-1].content)

AI: Your name is Thoufeeq. How can I help you today?


In [14]:
result3 = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="What did I tell you earlier?")
        ]
    },
    config=config
)

print("AI:", result3["messages"][-1].content)

AI: You told me that your name is Thoufeeq. Is there anything specific you would like to discuss or ask about?


#### Resume Previous Conversation

Even after new invocations, the thread retains all prior messages and the bot can reference them.

In [15]:
config_resume = {
    "configurable": {
        "thread_id": "session-1"
    }
}

result4 = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="Can you remember our previous conversation?")
        ]
    },
    config=config_resume
)

print("AI:", result4["messages"][-1].content)

AI: I can remember the context of our current conversation while we're chatting, but I don't have the ability to recall past interactions once the conversation ends. Each session is treated independently. How can I assist you further?


#### Inspect Full Thread State

Use `get_state()` to view all stored messages in the thread.

In [16]:
state = chatbot.get_state(config)

print("=== Conversation History ===")

for msg in state.values["messages"]:
    print(f"{msg.type}: {msg.content}")

=== Conversation History ===
human: My name is Thoufeeq.
ai: Nice to meet you, Thoufeeq! How can I assist you today?
human: What is my name?
ai: Your name is Thoufeeq. How can I help you today?
human: What did I tell you earlier?
ai: You told me that your name is Thoufeeq. Is there anything specific you would like to discuss or ask about?
human: Can you remember our previous conversation?
ai: I can remember the context of our current conversation while we're chatting, but I don't have the ability to recall past interactions once the conversation ends. Each session is treated independently. How can I assist you further?


#### Test Multiple Sessions

A different thread ID creates a separate conversation with its own fresh memory.

In [17]:
config2 = {
    "configurable": {
        "thread_id": "session-2"
    }
}

print("Thread 2 created")

Thread 2 created


In [18]:
result5 = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="My name is Rahul.")
        ]
    },
    config=config2
)

print("Thread-2 AI:", result5["messages"][-1].content)

Thread-2 AI: Nice to meet you, Rahul! How can I assist you today?


In [19]:
result6 = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?")
        ]
    },
    config=config2
)

print("Thread-2 AI:", result6["messages"][-1].content)

Thread-2 AI: Your name is Rahul. How can I help you today?


In [20]:
result7 = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?")
        ]
    },
    config=config
)

print("Session-1 AI:", result7["messages"][-1].content)

Session-1 AI: Your name is Thoufeeq. How can I help you today?


In [21]:
state1 = chatbot.get_state(config)

print("=== Session 1 ===")

for msg in state1.values["messages"]:
    print(f"{msg.type}: {msg.content}")

=== Session 1 ===
human: My name is Thoufeeq.
ai: Nice to meet you, Thoufeeq! How can I assist you today?
human: What is my name?
ai: Your name is Thoufeeq. How can I help you today?
human: What did I tell you earlier?
ai: You told me that your name is Thoufeeq. Is there anything specific you would like to discuss or ask about?
human: Can you remember our previous conversation?
ai: I can remember the context of our current conversation while we're chatting, but I don't have the ability to recall past interactions once the conversation ends. Each session is treated independently. How can I assist you further?
human: What is my name?
ai: Your name is Thoufeeq. How can I help you today?


In [22]:
state2 = chatbot.get_state(config2)

print("=== Session 2 ===")

for msg in state2.values["messages"]:
    print(f"{msg.type}: {msg.content}")

=== Session 2 ===
human: My name is Rahul.
ai: Nice to meet you, Rahul! How can I assist you today?
human: What is my name?
ai: Your name is Rahul. How can I help you today?


In [23]:
print("=== MEMORY TEST ===")

print("Session 1 messages:", len(state1.values["messages"]))
print("Session 2 messages:", len(state2.values["messages"]))

print()
print("Session 1 thread:", config["configurable"]["thread_id"])
print("Session 2 thread:", config2["configurable"]["thread_id"])

print()
print("Conversation memory test completed successfully.")

=== MEMORY TEST ===
Session 1 messages: 10
Session 2 messages: 4

Session 1 thread: session-1
Session 2 thread: session-2

Conversation memory test completed successfully.


#### Summary

Memory + persistence = stateful chatbot. The checkpointer saves state after every step, and thread IDs isolate conversations.